## Eydallin 2010 replica: glucose -> glycogen, host vs. host+clone, every screened gene

Eydallin et al. screened an *E. coli* AG1/DH1 genomic-fosmid library for glycogen effects
and reported 86 hits: `data/fabfos/benchmarks/eydallin/Y/measured_glycogen.tsv` carries
each gene's yield as a percent of wild-type glycogen.

`data/fabfos/runs/e_coli_dh1/gpr/gpr_gem.parquet` is the host's own GEM-GPR
evidence; `data/fabfos/runs/eydallin_clones/gpr/gpr_gem.parquet` carries one extra
`clone_gene` row per screened gene — the gene as it sits on the fosmid insert, keyed by
`condition_id` (`eydallin:glgC`, `eydallin:glgA`, ...). An overexpression clone is the host
row **and** the clone row both selected, so `ecspr.model.gpr` pools their conductances
together — never a special-cased "2x" argument to the solver.

Not every screened gene reaches the model: `clone_gem_census.tsv` records, per gene,
whether its ORF resolves in the host GEM at all, how many reactions the GEM assigns it,
and how many of those fall inside the atom-mapped bake universe this graph is built from.
This notebook classifies all 86 genes on those grounds before solving anything, runs the
two-point probe only on the genes that clear all three, and lists the rest with the
specific reason each was dropped.

In [8]:
from ecspr.model.graph import Terminal, solve
from ecspr.model.gpr import weights_from_rows
from ecspr.model.build import load_pairs, load_direction_ratios, graph_from_pairs

In [9]:
import sys
from pathlib import Path

import polars as pl

# ecspr's benchmark studies decode the bake into their own cache the first time they run
# (`bake_pairs.py`), keyed on the bake's own identity -- import it rather than re-deriving
# the decode here, so this notebook shares that cache instead of racing it.
ROOT = Path("/home/tony/agentic_workspace/projects/metasmith/fabfos/bench-eydallin")
sys.path.insert(0, str(ROOT / "research/fabfos/benchmarks"))
sys.path.insert(0, str(ROOT / "research/fabfos/benchmarks/eydallin"))
import bake_pairs  # noqa: E402

HOST_GPR = ROOT / "data/fabfos/runs/e_coli_dh1/gpr/gpr_gem.parquet"
CLONE_GPR = ROOT / "data/fabfos/runs/eydallin_clones/gpr/gpr_gem.parquet"
MEASURED = ROOT / "data/fabfos/benchmarks/eydallin/Y/measured_glycogen.tsv"

ELEMENT = "C"
GLUCOSE_MNXM = "MNXM1364061"     # D-glucose
GLYCOGEN_MNXM = "MNXM738130"     # glycogen

host = pl.read_parquet(HOST_GPR)
clones = pl.read_parquet(CLONE_GPR)
measured = pl.read_csv(MEASURED, separator="\t")
host.shape, clones.shape, measured.shape

((4886, 18), (81, 22), (86, 7))

In [ ]:
CENSUS_PATH = ROOT / "data/fabfos/runs/eydallin_clones/gpr/clone_gem_census.tsv"
HOST_DENOVO_GPR = ROOT / "data/fabfos/runs/e_coli_dh1/gpr/gpr_denovo.parquet"
CLONE_DENOVO_GPR = ROOT / "data/fabfos/runs/eydallin_clones/gpr/gpr_denovo.parquet"

census = pl.read_csv(CENSUS_PATH, separator="\t")
host_denovo = pl.read_parquet(HOST_DENOVO_GPR)
clones_denovo = pl.read_parquet(CLONE_DENOVO_GPR)

# clone_gem_census spells the ppk/glgP paralog "ppK"; measured_glycogen.tsv spells it "ppk".
# Rejoin onto the census spelling -- it is what n_reactions/n_in_universe below are keyed on.
measured = measured.with_columns(
    pl.when(pl.col("gene") == "ppk").then(pl.lit("ppK")).otherwise(pl.col("gene")).alias("gene")
)

# clone_gem_census only asks one question: does this gene's symbol already name a gene in
# the CURATED metabolic model (iECDH1ME8569)? A curated GEM is a claim about central
# metabolism and Eydallin's screen is genome-wide -- most of the 86 hits are regulators,
# transporters and y-genes the model was never built to represent, so a blank model_gene
# is a correct "not a metabolic gene", not a failed match (see build_clone_gpr.py's own
# docstring: "COVERAGE IS THE RESULT, NOT A PROBLEM"). build_clone_gpr_denovo.py is the
# other reading of the same clones: four sequence-annotation lanes run directly over the
# clones' own protein sequences -- resolved from the ASKA library download itself onto
# W3110 proteins in eydallin_clones.faa, not through the curated model at all -- and it
# reaches genes the curated model has no opinion on. It is folded in here as a fallback.
denovo_census = (
    clones_denovo
    .with_columns(pl.col("condition_id").str.strip_prefix("eydallin:").alias("gene"))
    .group_by("gene")
    .agg(
        pl.col("intermediate_id").n_unique().alias("denovo_n_reactions"),
        pl.col("mnxr").n_unique().alias("denovo_n_mnxr"),
        pl.col("mnxr").filter(pl.col("in_atom_universe")).n_unique()
          .alias("denovo_n_in_universe"),
    )
)

gene_status = (
    census.join(denovo_census, on="gene", how="left")
          .with_columns(pl.col("^denovo_.*$").fill_null(0))
)

# A gene is runnable through whichever channel gets it an atom-mapped reaction --
# curated preferred (it is the better-trusted annotation), de-novo as the fallback. Only a
# gene neither channel maps is genuinely out of reach for this graph. The reason columns
# record why EACH channel individually stops, so a dropped gene's cell can say why both did.
gene_status = gene_status.with_columns(
    pl.when(pl.col("n_in_universe") > 0).then(pl.lit("gem"))
      .when(pl.col("denovo_n_in_universe") > 0).then(pl.lit("denovo"))
      .otherwise(None)
      .alias("channel"),
    pl.when(pl.col("model_gene").is_null() | (pl.col("model_gene") == ""))
      .then(pl.lit("no ORF in the curated model matches this gene at all"))
      .when(pl.col("n_reactions") == 0)
      .then(pl.lit("the curated model's ORF for this gene carries zero reactions"))
      .otherwise(pl.lit("the curated reaction(s) exist but none are atom-mapped in the bake"))
      .alias("gem_reason"),
    pl.when(pl.col("denovo_n_mnxr") == 0)
      .then(pl.lit("no annotation lane assigns this ORF any reaction"))
      .otherwise(pl.lit("de-novo reaction(s) exist but none are atom-mapped in the bake"))
      .alias("denovo_reason"),
)

dropped = gene_status.filter(pl.col("channel").is_null())
ok_genes = (gene_status.filter(pl.col("channel").is_not_null())
                       .select(["gene", "channel"]).sort("gene"))

n_gem = int((ok_genes["channel"] == "gem").sum())
n_denovo = int((ok_genes["channel"] == "denovo").sum())
print(f"{ok_genes.height} of {gene_status.height} screened genes reach the graph "
      f"({n_gem} via the curated GEM, {n_denovo} only via the de-novo lanes); "
      f"{dropped.height} still dropped")
gene_status.group_by("channel").len().sort("channel")

In [ ]:
# show me all reactions of orf for glgC -- generalised: every gene neither channel
# reaches, and why each channel individually stopped on it.
for row in dropped.sort("gene").iter_rows(named=True):
    print(f"\n{row['gene']}")
    print(f"  GEM:     {row['gem_reason']}")
    print(f"  de-novo: {row['denovo_reason']}")

In [ ]:
LogOdds = False  # pool=True runs log-odds pooling (off by default) -- see ecspr_basic.ipynb

# Each channel is folded against its own background (curated vs. de-novo have different
# reaction sets), so the baseline solve and the weights below are per-channel too.
w_host_gem = weights_from_rows(host.to_pandas(), pool=LogOdds)
w_host_denovo = weights_from_rows(host_denovo.to_pandas(), pool=LogOdds)


def overexpressed_weights(gene, channel):
    condition_id = f"eydallin:{gene}"
    base, clone_tbl = (host, clones) if channel == "gem" else (host_denovo, clones_denovo)
    rows = clone_tbl.filter(pl.col("condition_id") == condition_id)
    # Pooling groups by (unit_id, channel, intermediate_id) -- and orf/unit_id are IDENTICAL
    # between the host row and the as-shipped clone row, since both name the same chromosomal
    # locus. Rekeying is what makes the fosmid-borne copy count as a physically separate
    # assertion, per ecspr.model.evidence's own account of what an overexpression is.
    rows = rows.with_columns(
        (pl.col("orf") + "::" + pl.col("condition_id")).alias("orf"),
        pl.col("condition_id").alias("unit_id"),
    )
    # host and clones don't share a column set (clones adds condition_id/cohort/...), so
    # align by name rather than by position.
    merged = pl.concat([base, rows], how="diagonal_relaxed").to_pandas()
    return weights_from_rows(merged, pool=LogOdds)


pairs = load_pairs(bake_pairs.atom_pairs(), element=ELEMENT)
ratios = load_direction_ratios(bake_pairs.direction_ratios())


def glucose_to_glycogen(weights):
    g = graph_from_pairs(pairs, ELEMENT, weights, ratios)
    src = Terminal.metabolite(g, GLUCOSE_MNXM, label="glucose")
    snk = Terminal.metabolite(g, GLYCOGEN_MNXM, label="glycogen")
    if src.missing or snk.missing:
        raise SystemExit(f"terminal missing: src={src.missing} snk={snk.missing}")
    return solve(g, src, snk)


sol_host_gem = glucose_to_glycogen(w_host_gem)
sol_host_denovo = glucose_to_glycogen(w_host_denovo)
sol_host_gem, sol_host_denovo

In [ ]:
from concurrent.futures import ProcessPoolExecutor


def _solve_one_gene(gene, channel):
    sol_host = sol_host_gem if channel == "gem" else sol_host_denovo
    sol_over = glucose_to_glycogen(overexpressed_weights(gene, channel))
    delta = sol_over.total - sol_host.total
    measured_row = measured.filter(pl.col("gene") == gene)
    return {
        "gene": gene,
        "channel": channel,
        "host_total": sol_host.total,
        "overexpressed_total": sol_over.total,
        "delta": delta,
        "delta_pct": 100 * delta / sol_host.total,
        "measured_pct_wt": measured_row["pct_wt"].item(0) if len(measured_row) else None,
    }


tasks = ok_genes.iter_rows(named=True)  # [{"gene": ..., "channel": ...}, ...]
genes, channels = zip(*((t["gene"], t["channel"]) for t in tasks))

# ProcessPoolExecutor forks after pairs/ratios/host*/clones*/w_host*/sol_host* are already
# built, so each worker inherits them for free -- only the gene/channel strings cross the
# pickle boundary per task, not the tables.
with ProcessPoolExecutor(max_workers=8) as pool:
    rows_out = list(pool.map(_solve_one_gene, genes, channels))

results = pl.DataFrame(rows_out).sort("delta_pct", descending=True)
results